In [1]:
%run Hilbert_C^2_package.ipynb
import numpy as np
import resource
from sage.rings.polynomial.polydict import ETuple
import datetime
import csv  # 用于CSV文件操作
import json
#import psutil
#import os

In [2]:
def create_laurent_ring(var_specs, base_ring=QQ):
    """
    生成指定变量个数的多变量洛朗多项式环（支持负次数）
    var_specs: 列表，每个元素为(变量前缀, 个数)，如[('x',2), ('y',3)]
    base_ring: 基域，默认有理数域QQ
    """
    variables = []
    # 按输入生成变量名（如x0,x1,y0,y1,y2）
    for prefix, count in var_specs:
        variables.extend([f"{prefix}{i}" for i in range(count)])
    # 创建洛朗多项式环，变量顺序与生成顺序一致
    R = LaurentPolynomialRing(base_ring, variables)
    # 返回环和对应的变量（方便直接使用）
    return R

'''# ------------------- 使用示例 -------------------
# 输入1：x要2个（x0,x1），y要3个（y0,y1,y2）
var_input = [('x', 2), ('y', 3)]
# 生成环R和变量x0,x1,y0,y1,y2
R = create_laurent_ring(var_input)

# 验证：创建含负次数的多项式
f = 2*x0^(-1)*x1^2 + 3*y0*y1^(-3)*y2  # 包含x0^-1、y1^-3等负次数项
print("变量列表:", R.gens())  # 输出：('x0', 'x1', 'y0', 'y1', 'y2')
print("含负次数的多项式f:", f)  # 输出：2*x0^-1*x1^2 + 3*y0*y1^-3*y2
print(f.dict())'''

'# ------------------- 使用示例 -------------------\n# 输入1：x要2个（x0,x1），y要3个（y0,y1,y2）\nvar_input = [(\'x\', 2), (\'y\', 3)]\n# 生成环R和变量x0,x1,y0,y1,y2\nR = create_laurent_ring(var_input)\n\n# 验证：创建含负次数的多项式\nf = 2*x0^(-1)*x1^2 + 3*y0*y1^(-3)*y2  # 包含x0^-1、y1^-3等负次数项\nprint("变量列表:", R.gens())  # 输出：(\'x0\', \'x1\', \'y0\', \'y1\', \'y2\')\nprint("含负次数的多项式f:", f)  # 输出：2*x0^-1*x1^2 + 3*y0*y1^-3*y2\nprint(f.dict())'

In [3]:
def save_result_to_csv(
    params, result, 
    filename="/Users/huajuan/Documents/文稿 - 花卷的MacBook Pro/computation of f_K/fKterm3运行结果记录.csv"):
    # 获取当前运行时间
    current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # 构造一行数据（与表头对应）
    row_data = {
        "recording time": current_time,
        "l(lambda)": params['p1'],
        "|lambda|": params['p2'],
        "l(lambda+)": params['p3'],
        "|lambda+|": params['p4'],
        "lambda-": result['r1'],
        "lambda+": result['r2'],
        "psi": params["p5"],
        "raw result": result['r3'],
        "normalized result": result['r4'],
        "result part1": result['r5'],
        "result part2": result['r6'],
        "guess result": result['r7'],
        "normalizer": params['p6']
    }
    
    # 检查文件是否存在：不存在则创建并写入表头，存在则直接追加行
    file_exists = False
    try:
        with open(filename, 'r', encoding='utf-8'):
            file_exists = True
    except FileNotFoundError:
        file_exists = False
    
    # 写入文件
    with open(filename, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=row_data.keys())
        if not file_exists:
            writer.writeheader()  # 首次运行写入表头
        writer.writerow(row_data)
    
    print(f"结果已保存到 {filename}")


### create_part2  definition
##### mapping is defined to be:
$$
k,r \mapsto \sum_{i=1}^k s_i z_i^r
$$

In [4]:
# convert mpoly.dict() back to a multivariable polynomial
# we didn't check whether the number of variables is aligned with which in the dictionary
# you need ensure this yourself
def load_dict(R, mpoly_dict):
    mpoly = sum([value * prod([R.gens()[i]^key[i] for i in range(len(key))])\
                for key, value in mpoly_dict.items()])
    return mpoly

# k, r --> \sum_1^k s_i*z_i^r
def create_part2(R, k, r):
    f = lambda i:tuple(1 if idx is i else 0 for idx in range(k)) + tuple(r if idx is i else 0 for idx in range(k))
    mpoly_dict = dict((f(idx), 1) for idx in range(k))
    res = load_dict(R, mpoly_dict)
    return res

# Return a polynomial with degree of s0,...,s_k-1 at most d whose j-th coefficients agree with self for all j<=d.
def truncate(f,k,R,d=2):
    truncated_dict = {key:value if all(deg <= d for deg in key[:k]) else 0 for key, value in f.dict().items()}
    res = load_dict(R, truncated_dict)
    return res

### create_fK_term2_part2  definition
$\lambda = (t2,t1)$ where $t1$ is $\lambda^+$, $t2$ is $\lambda^-$
##### mapping is defined to be:
$$
k,n,t1,t2,r_0 \mapsto \frac{r_0m_{-r_0}}{2}\left(\prod_{r\neq r_0}\left(\sum_{i=1}^k s_i z_i^r\right)^{m_{-r}}\right)\cdot \left(\sum_{i=1}^k s_i z_i^{r_0}\right)^{m_{-r_0}-1}\cdot \sum_{i=1}^k s_i^2z_i^{r_0}
$$

In [5]:
def create_fK_term2_part2(R,k,n,t1,t2,r0):
    # 计算需要统一的目标长度（取两者最大值）
    max_len = max(len(t1), len(t2))
    
    # 补0并转换为元组（切片保证不超出原长度，*[0]补够剩余长度）
    t1_padded = tuple(list(t1) + [0] * (max_len - len(t1)))
    t2_padded = tuple(list(t2) + [0] * (max_len - len(t2)))
    
    if r0 > 0:
        # define part 2 of expression of f_0,lambda^k
        fK_pos = (
            prod([create_part2(R, k, i+1)^t2_padded[i] if i != r0-1 else 1 for i in range(max_len)]) *
            create_part2(R, k, r0)^(t2_padded[r0-1]-1)
            * sum([R.gens()[idx]^2 * R.gens()[idx+k]^r0 for idx in range(k)]) * 1/2 * r0 * t2_padded[r0-1])
        #print(fK_pos)
        fK_neg = prod([create_part2(R, k, -i-1)^t1_padded[i] for i in range(max_len)])
        #print(fK_neg)
        fK_term1_part2 = fK_neg * fK_pos * R.gens()[k]^-n
        #print(fK_part2)
        return fK_term1_part2
    else:
        # define part 2 of expression of f_0,lambda^k
        fK_pos = prod([create_part2(R, k, i+1)^t2_padded[i] for i in range(max_len)])
        #print(fK_pos)
        fK_neg = (
            prod([create_part2(R, k, -i-1)^t1_padded[i] if i != -r0-1 else 1 for i in range(max_len)]) * 
            create_part2(R, k, r0)^(t1_padded[-r0-1]-1) * 
            sum([R.gens()[idx]^2 * R.gens()[idx+k]^r0 for idx in range(k)]) * 1/2 * r0 * t1_padded[-r0-1]
        )
        #print(fK_neg)
        fK_term1_part2 = truncate(fK_neg * fK_pos * R.gens()[k]^-n,k,R)
        #print(fK_part2)
        return fK_term1_part2

In [6]:
def get_constant_term(f1,f2):
    # Compute constant term of product of two multivariate Laurent polynomials
    res = 0
    for mono, coeff in f1.dict().items():
        if ETuple((-np.array(mono)).tolist()) in f2.dict():
            temp = coeff * f2.dict()[ETuple((-np.array(mono)).tolist())]
            res += temp
    return res

In [7]:
def get_para_constant(f1, f2, R):
    # 计算两个多元Laurent多项式的条件多项式：
    # - 总变量数默认是 2k+1（最后1个是a）
    # - 前2k个变量：f1的指数 = -f2的指数（乘积后次数为0）
    # - a的指数：不抵消，直接累加（f1_a_exp + f2_a_exp）
    # - 输出：关于a的多项式（所有匹配项累加，自动合并同类项）
    
    res = 0  
    for mono1, coeff1 in f1.dict().items():
        # 拆分f1的单项式：前n-1个变量的指数 + a的指数
        mono1_first = mono1[:-1]  # 前2k个普通变量的指数
        a_exp1 = list(mono1)[-1]        # f1中a的指数
        #print('------------')
        #print('mono1: ', mono1, a_exp1)
        # 构造f2需要匹配的「前2k个变量的指数」（互为相反数）
        target_first = (-np.array(mono1_first)).tolist()
        
        # 遍历f2的所有单项式，寻找「前2k个指数匹配」的项（不管a的指数）
        for mono2, coeff2 in f2.dict().items():
            mono2_first = mono2[:-1]  # f2中前2k个普通变量的指数
            a_exp2 = list(mono2)[-1]        # f2中a的指数
            #print('mono2: ', mono2, a_exp2)
            #print('test: ', ETuple(mono2_first), ETuple(target_first), ETuple(mono2_first) == ETuple(target_first))
            # 核心判断：前2k个变量的指数是否互为相反数
            if ETuple(mono2_first) == ETuple(target_first):
                # 累加项：系数乘积 * a^(a_exp1 + a_exp2)
                res += coeff1 * coeff2 * (R.gens()[-1]^(a_exp1 + a_exp2))
                #print(R.gens()[-1])
                #print(coeff1, coeff2, a_exp1, a_exp2, (R.gens()[-1]^(a_exp1 + a_exp2)))
                #print('test res: ', res)
    
    return res

### temp1  definition
##### mapping is defined to be:
$$
j, maxlen \mapsto \sum_{i=1}^{j-1} s_i\left(\sum_{l=-maxlen}^{maxlen} lz_i^lz_j^{-l}\right)
$$

In [8]:
# define part 1 of expression of f_0,lambda^k
def temp1(j,s_neg):
    # \sum_1^j-1 s_i * (\sum_(-maxlen)^maxlen l * z_i^l * z_j^-l)
    f = (
        lambda i:R.gens()[i]*
        sum([l * R.gens()[i+k]^l * R.gens()[j+k-1]^(-l)
             - l * R.gens()[i+k]^(-l) * R.gens()[j+k-1]^l for l in range(s_neg+1)])
    )
    res = sum([f(i) for i in range(j-1)])
    return res
'''j=3
test = temp1(j,max_len)
print(test)'''

'j=3\ntest = temp1(j,max_len)\nprint(test)'

### temp2  definition
##### mapping is defined to be:
$$
j, maxlen \mapsto \sum_{i=1}^{j-1} s_i\left(\frac12s_i\sum_{l=-maxlen}^{maxlen} p^{\sigma(l)}l^2z_i^lz_j^{-l}-\frac12s_j\sum_{l=-maxlen}^{maxlen} p^{1-\sigma(l)}l^2z_i^lz_j^{-l}\right)
$$

In [9]:
# define part 1 of expression of f_0,lambda^k
def temp2(j,s_neg):
    # \sum_1^j-1 s_i * (\sum_(-maxlen)^maxlen l * z_i^l * z_j^-l)
    f = (
        lambda i:R.gens()[i]*
        sum([l^2 * R.gens()[i+k]^l * R.gens()[j+k-1]^(-l)
             + R.gens()[-1] * l^2 * R.gens()[i+k]^(-l) * R.gens()[j+k-1]^l for l in range(s_neg+1)])
    )
    g = (
        lambda i:R.gens()[i]*
        sum([R.gens()[-1] * l^2 * R.gens()[i+k]^l * R.gens()[j+k-1]^(-l)
             + l^2 * R.gens()[i+k]^(-l) * R.gens()[j+k-1]^l for l in range(s_neg+1)])
    )
    res1 = sum(
        [1/2 * f(i) * R.gens()[i] for i in range(j-1)]
    )
    res2 = sum(
        [ - 1/2 * g(i) * R.gens()[j-1] for i in range(j-1)]
    )
    return [res1, res2]

In [10]:
guess_res = (
    lambda k,n,t1,t2:
    -1/2*n*factorial(k-1)*n^k
    + (sum([(r+1-n)^k*t2[r] for r in range(len(t2))])
    + sum([(-r-1-n)^k*t1[r] for r in range(len(t1))]))*(-1)^k*1/2*factorial(k-2)*n
)

In [11]:
def fK_term3_part2(psi):
    part1 = prod([temp1(j,s_neg) if j != psi else 1 for j in range(2, k+1)])
    res1 = part1 * temp2(psi, s_neg)[0]
    res2 = part1 * temp2(psi, s_neg)[1]
    return [res1,res2]

In [25]:
# Basic settings
k = 3
n = 3
l_pos = 2
s_pos = 6

var_input = [('s', k), ('z', k), ('p', 1)]
R = create_laurent_ring(var_input)

l_neg = k - l_pos
s_neg = s_pos + n
P_pos_list = Partitions(s_pos, length = l_pos)
P_neg_list = Partitions(s_neg, length = l_neg)
lam_pos = P_pos_list.random_element()
lam_neg = P_neg_list.random_element()
print(lam_neg,lam_pos)

lam_pos_mul = partition_to_multiplicity(lam_pos)
lam_neg_mul = partition_to_multiplicity(lam_neg)
print(lam_neg_mul,lam_pos_mul)
#f = load_dict(R, {(-1, 2, 0, 0, 0): 2, (0, 0, 1, -3, 1): 3})
#print(f)
#f2 = create_part2(R, k, 3)
#print(f2)

# 统一两个tuple的长度，不足的部分用0补齐
# 定义两个原始元组
t1 = lam_pos_mul
t2 = lam_neg_mul

# 计算需要统一的目标长度（取两者最大值）
max_len = max(len(t1), len(t2))

# 补0并转换为元组（切片保证不超出原长度，*[0]补够剩余长度）
t1_padded = tuple(list(t1) + [0] * (max_len - len(t1)))
t2_padded = tuple(list(t2) + [0] * (max_len - len(t2)))
print(t2_padded,t1_padded)

f_pos = prod([create_part2(R, k, i+1)^t2_padded[i] for i in range(max_len)])
f_neg = prod([create_part2(R, k, -i-1)^t1_padded[i] for i in range(max_len)])
f_mod = n*R.gens()[k]^(-n)/(prod([R.gens()[i]^2 for i in range(k)]))

fK3_main1 = lambda psi, ell:(
    prod([temp1(j,s_neg) if j != psi else 1 for j in range(2, k+1)])
    * sum([R.gens()[i]^2 * R.gens()[i+k]^ell * R.gens()[psi+k-1]^(-ell) for i in range(psi-1)])
)
fK3_main2 = lambda psi, ell:(
    prod([temp1(j,s_neg) if j != psi else 1 for j in range(2, k+1)])
    * sum([R.gens()[i] * R.gens()[psi-1] * R.gens()[i+k]^ell * R.gens()[psi+k-1]^(-ell) for i in range(psi-1)])
)
#print(f_mod,f_neg,f_pos)

[9] [5, 1]
(0, 0, 0, 0, 0, 0, 0, 0, 1) (1, 0, 0, 0, 1)
(0, 0, 0, 0, 0, 0, 0, 0, 1) (1, 0, 0, 0, 1, 0, 0, 0, 0)


In [22]:
# multiply by p = 2 - a when l is negative
'''for ell in list(range(- s_neg - 1, 0)) + list(range(1, s_neg + 2)):
    res = get_constant_term(f_mod*f_neg*f_pos, sum([fK3_main1(psi, ell) for psi in range(2, k+1)]))
    print(f"l: {ell}, result1 = {res}, raw result1 = {res*ell^2/2}")'''
for psi in range(2, k+1):
    print('----------------------')
    for ell in list(range(- s_neg - 1, 0)) + list(range(1, s_neg + 2)):
        res = get_constant_term(f_mod*f_neg*f_pos, fK3_main1(psi, ell))
        if res != 0:
            print(f"psi: {psi}, l: {ell}, result1 = {res/n}, raw result1 = {res*ell^2/n}")

----------------------
psi: 2, l: 3, result1 = 6, raw result1 = 54
----------------------
psi: 3, l: -4, result1 = -4, raw result1 = -64
psi: 3, l: 4, result1 = 4, raw result1 = 64
psi: 3, l: 6, result1 = 6, raw result1 = 216


In [23]:
# multiply by p = 2 - a when l is positive
for psi in range(2, k+1):
    print('----------------------')
    for ell in list(range(- s_neg - 1, 0)) + list(range(1, s_neg + 2)):
        res = get_constant_term(f_mod*f_neg*f_pos, fK3_main2(psi, ell))
        if res != 0:
            print(f"psi: {psi}, l: {ell}, result2 = {res/n}, raw result2 = {-n*res*ell^2/2}")

----------------------
psi: 2, l: -4, result2 = -4, raw result2 = 288
psi: 2, l: -3, result2 = 6, raw result2 = -243
psi: 2, l: -1, result2 = 4, raw result2 = -18
psi: 2, l: 4, result2 = 4, raw result2 = -288
psi: 2, l: 6, result2 = 6, raw result2 = -972
psi: 2, l: 7, result2 = -4, raw result2 = 882
----------------------
psi: 3, l: -3, result2 = 6, raw result2 = -243
psi: 3, l: -1, result2 = 6, raw result2 = -27
psi: 3, l: 7, result2 = 6, raw result2 = -1323


In [26]:
f_pos = prod([create_part2(R, k, i+1)^t2_padded[i] for i in range(max_len)])
f_neg = prod([create_part2(R, k, -i-1)^t1_padded[i] for i in range(max_len)])
fK_term3_part1 = f_neg * f_pos * R.gens()[k]^-n

save = False
for psi in range(2, k+1):
    part1 = n*prod([R.gens()[i]^-2 for i in range(k)]) * fK_term3_part1
    res1 = get_para_constant(part1, fK_term3_part2(psi)[0], R)
    res2 = get_para_constant(part1, fK_term3_part2(psi)[1], R)
    res = res1 + res2
    normalizer = n/2
    print('------')
    #print('part1:', part1)
    #print('part2: ', 2*fK_term3_part2(psi))
    print('psi = ', psi)
    print('res is', res)
    print('res1 is', res1/normalizer)
    print('res2 is', res2/normalizer)
    print('normalized res is', res/normalizer)
    #print('guess result is', -guess_res(k,n,t1_padded,t2_padded))
    if save == True:
        # 调用保存函数
        save_result_to_csv(
            params={"p1": k, "p2": n, "p3": l_pos, "p4": s_pos, "p5": psi, "p6": normalizer},
            result={
                "r1": lam_neg, 
                "r2": lam_pos, 
                "r3": json.dumps((str(res))), 
                "r4": json.dumps((str(res/normalizer))),
                "r5": json.dumps((str(res1/normalizer))), 
                "r6": json.dumps((str(res2/normalizer))),
                #"r7": -guess_res(k,n,t1_padded,t2_padded)
                   }
        )

------
psi =  2
res is -135*p0 + 99
res1 is 54
res2 is -90*p0 + 12
normalized res is -90*p0 + 66
------
psi =  3
res is -1053*p0 + 630
res1 is -216*p0 + 576
res2 is -486*p0 - 156
normalized res is -702*p0 + 420


In [ ]:
#上面是直接计算
#-----------------
#下面是通过展开原来的式子计算

In [ ]:
def find_valid_tuples_general(power_pattern, max_index):
    """
    【超级通用版】寻找满足次数约束的元组 (w2,w3,w4,...)
    输入：
        power_pattern: 幂次规则，字典形式
            例1：{1:2, 'w2':1, 'w3':1, 'w4':1} → s1² s_w2 s_w3 s_w4
            例2：{1:1, 'w2':2, 'w3':1, 'w4':1} → s1 s_w2² s_w3 s_w4
            例3：{1:2, 'w2':1, 'w3':1, 'w4':1, 'w5':1} → 扩展到 w5
        max_index: 最大下标，如 4 → 到 w4；5 → 到 w5
    输出：
        所有合法元组 (w2,w3,w4,...)
    """
    from itertools import product

    # 自动生成变量列表 [w2,w3,...wmax]
    var_list = [f'w{i}' for i in range(2, max_index+1)]
    
    # 自动生成每个 w_i 的范围：1<=w_i<=i
    ranges = [range(1, i+1) for i in range(2, max_index+1)]
    
    valid = []
    
    # 遍历所有可能组合
    for values in product(*ranges):
        # 映射：w2→v2, w3→v3,...
        val_map = {var: val for var, val in zip(var_list, values)}
        
        # 初始化次数
        deg = {}
        
        # 统计每一项的次数
        for key, p in power_pattern.items():
            if key == 1:
                # 固定项 s1
                j = 1
            else:
                # 变量项 sw_i → 取当前值
                j = val_map[key]
            
            if j in deg:
                deg[j] += p
            else:
                deg[j] = p
        
        # 检查：所有次数 ≤ 2
        if all(d <= 2 for d in deg.values()):
            valid.append(values)
    
    return valid

def get_u(pattern, valid_tuple):
    max_index = len(valid_tuple) + 2
    res = [2] * max_index
    
    for key, val in pattern.items():
        if key == 1:
            # 纯数字 key：1 → 位置 1
            idx = key - 1
        else:
            # 字符串 key：'w2' → 提取数字 2
            idx = valid_tuple[int(key[1:]) - 2] - 1
        
        # 安全判断：不越界
        if 0 <= idx <= max_index:
            res[idx] -= val
    
    return res

In [ ]:
def build_weight_matrix(w_tuple):
    """
    生成你要的 k×k 矩阵：第j列 wj行~j行=1，其余=0
    """
    k = len(w_tuple)
    M = matrix(ZZ, k, k)  # 整数环零矩阵
    
    for j in range(1, k+1):
        wj = w_tuple[j-1]
        start = wj
        end = j
        for i in range(start, end+1):
            M[i-1, j-1] = 1
    return M


def compute_inverse_times_vector(w_tuple):
    """
    主函数：
    1. 生成矩阵
    2. 求逆
    3. 生成向量 (d2+...+d_{k+1}, ..., d_{k+1})
    4. 返回 逆矩阵 × 向量
    """
    k = len(w_tuple)
    M = build_weight_matrix(w_tuple)
    
    # 求逆矩阵（Sage自动计算）
    M_inv = M.inverse()
    
    # 定义符号 d2, d3, ..., d_{k+1}
    vars_list = [var(f'd{i}') for i in range(2, k+2)]  # [d2,d3,...,d_{k+1}]
    
    # 生成你要的向量：(d2+...+d_{k+1}, ..., d_k+d_{k+1}, d_{k+1})
    vec = []
    for i in range(k):
        # 第i项：d_{2+i} + ... + d_{k+1}
        term = sum(vars_list[i:])
        vec.append(term)
    vec = vector(vec)
    
    # 计算 逆矩阵 × 向量
    result = M_inv * vec
    
    return [M, M_inv, vec, result]

In [ ]:
def get_r_deg(u_tuple):
    k = len(u_tuple)
    var_list = [var(f'r{i}') for i in range(1, k+1)]
    r_vec = [0] * k
    r_vec[0] = sum(var_list[:u_tuple[0]])
    for i in range(1,k):
        r_vec[i] = sum(var_list[:sum(u_tuple[:i+1])]) - sum(var_list[:sum(u_tuple[:i])])
    return r_vec

def compute_final_vector(w_tuple, u_tuple):
    """
    主函数：
    1. 生成矩阵
    2. 求逆
    3. 生成向量 (d2+...+d_{k+1}, ..., d_{k+1})
    4. 返回 逆矩阵 × 向量
    """
    k = len(w_tuple) + 1
    M = build_weight_matrix((1,) + w_tuple)
    # 求逆矩阵（Sage自动计算）
    M_inv = M.inverse()
    
    # 定义deg (r1+,..., ..+r_k)
    r_list = get_r_deg(u_tuple)
    
    # 生成你要的向量：(d2+...+d_{k+1}, ..., d_k+d_{k+1}, d_{k+1})
    vec = [sum(r_list[i+1:]) for i in range(k)]
    vec = vector(vec)
    #print(vec)
    # 计算 逆矩阵 × 向量
    result = M_inv * vec
    
    return result

In [ ]:
u_tuple = u_list[0]
w_tuple = w_list[0]
print(get_r_deg(u_tuple)) 
print(compute_final_vector(w_tuple,u_tuple))

In [ ]:
# 定义幂次模式
pattern = {1:2, 'w2':1, 'w3':1, 'w4':1, 'w5':1}

w_list = find_valid_tuples_general(pattern, max_index=5)
print("合法元组：", w_list)
u_list = [get_u(pattern, word) for word in w_list]
print('u obeys: ', u_list)
deg_list = [compute_inverse_times_vector((1,) + word)[-1] for word in w_list]
fin_list = [compute_final_vector(w,u) for w, u in zip(w_list,u_list)]
for word in w_list:
    # 输入你的元组 (w1,w2,w3,w4)
    w_tuple = (1,) + word

    # 计算
    M, M_inv, vec, res = compute_inverse_times_vector(w_tuple)

    # 输出
    '''print("=== 生成的矩阵 ===")
    print(M)

    print("\n=== 逆矩阵 ===")
    print(M_inv)

    print("\n=== 你的向量 ===")
    print(vec)
    '''
    print("\n=== 逆矩阵 × 向量 结果 ===")
    print(res)
    
for w,u in zip(w_list,u_list):
    print('\n===r_vec result is ===')
    print(compute_final_vector(w,u))

In [ ]:
from IPython.display import display, HTML

def show_clean_table(w_list, u_list, result_list, fin_list):
    html = '''
    <style>
        .custom-table {
            font-family: Arial, sans-serif;
            border-collapse: collapse;
            width: 100%;
            font-size: 14px;
        }
        .custom-table th, .custom-table td {
            border: 1px solid #ddd;
            padding: 8px 12px;
            text-align: left;
        }
        .custom-table th {
            background-color: #f5f5f5;
        }
    </style>
    <table class="custom-table">
        <tr>
            <th>w_tuple</th>
            <th>u_tuple</th>
            <th>result</th>
            <th>r_deg</th>
        </tr>
    '''
    
    for w, u, res, fin in zip(w_list, u_list, result_list, fin_list):
        html += f"<tr><td>{w}</td><td>{u}</td><td>{res}</td><td>{fin}</td></tr>"
    
    html += "</table>"
    display(HTML(html))

show_clean_table(w_list, u_list, deg_list, fin_list)

In [ ]:
# test
r_par = [-5,-1,-1,2,2]
r_par_list = [sum(p) for p in combinations(r_par,2)]
r_res = [6*n^2*(n+s)^2*(-s) + 4*n^2*s^2*(n+s) + 2*n^2*(2*n+s)*(n+s)^2 for s in r_par_list]
print("result is :", sum(r_res))

In [ ]:
# test
from itertools import combinations
# guess results
# 1. 定义 sigma 函数：负数返回1，0和正数返回0
def sigma(x):
    """
    自定义符号函数：
    - 输入x < 0 → 返回 1
    - 输入x = 0 或 x > 0 → 返回 0
    """
    if x < 0:
        return 1
    else:
        return 0

# 2. 定义一元多项式环：变量为p，系数域为有理数域Q
Guess_R.<pp> = PolynomialRing(QQ)

def calculate_polynomial(par, r_length, r_deg):
    """
    计算公式 2 * sum_{i<j<k} p^sigma(-r_i - r_j - r_k) * (r_i + r_j + r_k)^4
    其中：
    - r_i/r_j/r_k ∈ {1,2,3,4} 且 i<j<k
    - sigma函数：负数返回1，0/正数返回0
    返回关于p的一元多项式
    """
    total = 0
    # 生成所有满足i<j<k的三元组（1-4中选3个不重复的数）
    for r in combinations(par, r_length):
        s = -sum(r)          # 计算-(r_i + r_j + r_k)
        sigma_val = sigma(s)     # 计算sigma(-(r_i+r_j+r_k))
        term = (pp^sigma_val) * (s^r_deg)  # 计算每一项：p^sigma值 * 和的4次方
        total += term
        
    return total

# 3. 调用函数并输出结果
par = [-5,-1,-1,2,2]
'''
poly = calculate_polynomial(par, 3, 4)
print("最终的一元多项式：")
print(poly)
'''
guess_res = (
    24*n^5 + 
    20*n^2*calculate_polynomial(par, 2, 3) - 20*n*calculate_polynomial(par, 2, 4) + 6*calculate_polynomial(par, 2, 5)
    + 10*n*calculate_polynomial(par, 3, 4) - 6*calculate_polynomial(par, 3, 5)
    + 6*calculate_polynomial(par, 4, 5)
)
#2*calculate_polynomial(par, 3, 4) + 3*calculate_polynomial(par, 2, 3) - calculate_polynomial(par, 2, 4)
print("guess result is")
print(guess_res)